<h1 style="
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    font-size: 36px;
    color: #ffffff;
    background-color: #9E1B32;
    padding: 20px;
    border-radius: 12px;
    text-align: center;
    box-shadow: 0px 4px 10px rgba(0, 0, 0, 0.1);">
    TEEHR for NextGen Model Output Analysis and Evaluation
</h1>

**Authors:** 

<ul style="line-height:1.5;">
<li>Savalan Naser Neisary <a href="mailto:snaserneisary@crimson.ua.edu">(snaserneisary@crimson.ua.edu)</a></li>
</ul>


**Last Updated:** 10/10/2026

**Purpose:**

**TEEHR** (**T**ools for **E**xploratory **E**valuation in **H**ydrologic **R**esearch) is a Python package that provides a flexible framework for evaluating hydrologic model performance. In this notebook, **TEEHR** is used to assess NextGen model outputs by comparing simulated streamflow with `USGS observations` and the `National Water Model Retrospective Analysis v3.0`.
The workflow demonstrates how to set up the TEEHR evaluation environment, organize required datasets, and generate performance metrics and diagnostic plots. More information about **TEEHR** can be found in the 
<span style="background-color:rgba(184,34,61,0.12); padding:4px 6px; border-radius:6px;">
  <a href="https://rtiinternational.github.io/teehr/user_guide/index.html#user-guide" target="_blank" style="font-weight:bold; color:#E04A63;">
    TEEHR User Guide
  </a>
</span>.

**Audience:**

Researchers and practitioners working with NextGen hydrologic simulations. Users should be familiar with Python, Jupyter Notebooks, and basic hydrologic modeling concepts.

**Description:**

<div style="font-size:14px; margin-bottom:0; padding-bottom:0;">

This notebook leverages **TEEHR** to load, organize, and evaluate hydrologic datasets needed to assess NextGen streamflow simulations.  
It performs the following tasks:
- Reads `NextGen-simulated streamflow` outputs.  
- Aligns them with `USGS observed discharge` and `NWM v3.0 retrospective simulations`.  
- Sets up or clones a `TEEHR evaluation directory`.  
- Computes performance metrics and generates diagnostic plots for model evaluation.
</div>
<div style="margin-top:-16px;"></div>

**Data Description:**

This notebook uses three hydrologic datasets organized according to **TEEHR’s** primary–secondary evaluation structure:

**1. USGS Observed Streamflow** (`Primary Dataset`): Ground-truth discharge time series from the U.S. Geological Survey (NWIS).  
This dataset serves as the **primary reference** against which all model outputs are evaluated.

**2. NextGen Model Outputs** (`Secondary Dataset`): Simulated streamflow generated by the **NextGen hydrologic model** for each evaluation location. Loaded in TEEHR as a **secondary dataset**.

**3. NWM Retrospective Analysis v3.0** (`Secondary Dataset`): Simulated streamflow from the `National Water Model v3.0 retrospective dataset`.  
Also treated as a **secondary dataset**.

Supporting metadata (gage identifiers, attributes, evaluation configuration files) ensures proper `spatial and temporal alignment` across datasets.  

<div style="padding: 15px 20px; background-color: rgba(184,34,61,0.12); border-left: 6px solid #9E1B32; border-radius: 4px; margin-bottom: 10px;">
<h3 style="margin-top: 0; font-weight: 700;">❗ We created this tutorial based on the tutorial in the <a href="https://www.hydroshare.org/resource/27045581bdea4808a393330f2417379c/" target="_blank"><strong>NextGen Water Modeling Workflow for Research-Scale Applications Hydroshare resources</strong></a>.</h3>

Nassar, A., Tarboton, D., Baig, F., Patel, A., Cunningham, J., Salehabadi, H. (2026). NextGen Water Modeling Workflow for Research-Scale Applications, HydroShare, https://doi.org/10.4211/hs.27045581bdea4808a393330f2417379c, DOI for this published resource is pending activation.
</div>

<div style="background:#9E1B32; border-left:6px solid #F0627A; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#fff5f7;">
    1. Prepare the Python Environment
  </h3>
</div>

We prepare the **Python environment** by importing all required libraries and utilities used throughout this TEEHR evaluation workflow, which supports **data processing**, **time-series visualization**, and **NextGen/TEEHR evaluation**.

In [ ]:
import shutil
from pathlib import Path
from typing import Union
import re
import logging
from dataretrieval import nwis
import geopandas as gpd
from shapely.geometry import Point
import requests

import pandas as pd
import xarray as xr
import hvplot.pandas
from datetime import datetime

from bokeh.io import output_notebook
output_notebook()

from teehr.evaluation.utils import print_tree
import teehr
from utils import ngiab_utils

from utils.workshop_paths import DATA_DIR

from utils.data_prep_plots import register_usgs_gage, plot_teehr_streamflow

from teehr import DeterministicMetrics as dm

logger = logging.getLogger(__name__)

print(f"TEEHR version: {teehr.__version__}")

<div style="background:#9E1B32; border-left:6px solid #F0627A; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#fff5f7;">
    2. Set Inputs
  </h3>
</div>

<p style="margin-top:0; padding-top:4px;">
In this step, you define the <strong>key inputs</strong> required for the TEEHR workflow.  
These inputs specify the <strong>hydrofabric domain</strong> you are working with and the <strong>directories used for preprocessing and data preparation</strong>.
</p>

<div style="background-color:rgba(184,34,61,0.12); border-left: 5px solid #E04A63; padding: 14px; border-radius: 8px; font-family: 'Segoe UI', sans-serif; font-size: 15px; margin-top: 10px;">
  <strong>Inputs Defined in the Next Cell:</strong>
  <ul style="margin-top: 8px; margin-bottom: 0;">
    <li><strong>Hydrofabric ID:</strong> Specifies the spatial domain used for this NGIAB run.  
      In this example, the domain is a USGS gage:
      <ul>
        <li><code>gage-02450250</code></li>
      </ul>
    </li>
    <li><strong>Preprocessing Directory:</strong>  
      The path to the NGIAB preprocessing output associated with the selected hydrofabric ID.
      <br><em>Example:</em>  
      <code>/home/jovyan/ngiab_preprocess_output/gage-02450250</code>
    </li>
    <li><strong>Temporary Output Directory:</strong>  
      This folder stores the TEEHR evaluation files, including metadata, time series, and supporting scripts.
      <br><em>Example:</em>  
      <code>/home/jovyan/ngiab_preprocess_output/gage-02450250/teehr</code>
    </li>
  </ul>
</div>

In [ ]:
# Hydrofabric/gage ID for this NGIAB run
hydrofabric_id = "gage-02450250"

# Path to the NGIAB preprocessing output for this gage
SAMPLE_NGIAB_OUTPUT_DIR = Path(f"{DATA_DIR}/{hydrofabric_id}")

# Directory containing all TEEHR evaluation data and supporting files.
TEMP_DIR = Path(f"{DATA_DIR}/{hydrofabric_id}/teehr")

# Create the temp directory if it doesn't exist
TEMP_DIR.mkdir(parents=True, exist_ok=True)

# If we want the code to be online. If yes, it will take longer. 
online = 'yes'

SAMPLE_NGIAB_OUTPUT_DIR, TEMP_DIR

<div style="background:#9E1B32; border-left:6px solid #F0627A; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#fff5f7;">
    3. Get a Table Mapping NGIAB IDs to USGS Gage IDs
  </h3>
</div>

<p style="margin-top:0; padding-top:4px;">
This step extracts the <strong>USGS gage identifiers</strong> from the hydrofabric and generates a lookup table that links each 
<strong>NGIAB segment ID</strong> (<code>ngen</code>) to its corresponding <strong>USGS gage ID</strong> to ensure that discharge records from NGIAB outputs are aligned with observed USGS streamflow data.
</p>

<div style="background-color:rgba(184,34,61,0.12); border-left: 5px solid #E04A63; padding: 14px; border-radius: 8px; font-family: 'Segoe UI', sans-serif; font-size: 15px; margin-top: 10px;">
  <strong>What Happens in This Step:</strong>
  <ul style="margin-top: 8px; margin-bottom: 0;">
    <li>The hydrofabric is queried to extract all gages associated with the selected domain.</li>
    <li>A correspondence table is created linking:
      <ul>
        <li><strong>NGIAB routing IDs</strong> (model segment identifiers)</li>
        <li><strong>USGS gage IDs</strong> (e.g., <code>02465493</code>)</li>
      </ul>
    </li>
      </ul>
    </li>
  </ul>
</div>

In [ ]:
# Extract gage information from the hydrofabric directory.
# This returns a list of tuples in the format: (ngen_segment_id, usgs_gage_id)
ngiab_usgs_gages = ngiab_utils.get_gages_from_hydrofabric(SAMPLE_NGIAB_OUTPUT_DIR)

# Convert the list of gages into a clean Pandas DataFrame
ngiab_gages_df = pd.DataFrame(ngiab_usgs_gages, columns=["ngen", "usgs"])

# Standardize USGS gage ID format by adding a "usgs-" prefix
# Example: 123  →  usgs-123
ngiab_gages_df["usgs"] = "usgs-" + ngiab_gages_df["usgs"].astype(str)

# Normalize NGIAB routing segment names by replacing "wb" with "ngen"
# Example: wb-123  →  ngen-123
ngiab_gages_df["ngen"] = ngiab_gages_df["ngen"].str.replace("wb", "ngen")

# Display the final crosswalk table
# This links each NGIAB segment (ngen) to its corresponding USGS gage ID
ngiab_gages_df

<div style="background:#9E1B32; border-left:6px solid #F0627A; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#fff5f7;">
    4. Get the NGIAB Simulation Start and End Times
  </h3>
</div>
<p style="margin-top:0; padding-top:6px;">
This step reads the NGIAB configuration files to extract the simulation’s 
<strong>start</strong> and <strong>end timestamps</strong> to define the valid time window for the model outputs and ensure that all subsequent 
processing steps operate within the correct temporal bounds.
</p>

<div style="background-color:rgba(184,34,61,0.12); border-left: 5px solid #E04A63; padding:14px; border-radius:8px;
            font-family:'Segoe UI', sans-serif; font-size:15px; margin-top:10px;">
  <strong>What Happens in This Step:</strong>
  <ul style="margin-top:8px; margin-bottom:0;">
    <li>The workflow searches the NGIAB preprocessing directory for the model’s time configuration file.</li>
    <li>It extracts the defined <strong>simulation start</strong> and <strong>end times</strong> as Python datetime objects.</li>
  </ul>
</div>

In [ ]:
start_date, end_date = ngiab_utils.get_simulation_start_end_time(
    SAMPLE_NGIAB_OUTPUT_DIR
)
start_date, end_date

<div style="background:#9E1B32; border-left:6px solid #F0627A; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#fff5f7;">
    5. Create an Evaluation Object and Directory Using an Empty Template
  </h3>
</div>
<p style="margin-top:0; padding-top:6px;">
We create an empty <strong>TEEHR Evaluation</strong> object along with a dedicated 
directory where all evaluation inputs, outputs, and configuration files will be stored to provide a structured workspace for organizing the results.
</p>

<div style="background-color:rgba(184,34,61,0.12); border-left: 5px solid #E04A63; padding:14px; border-radius:8px;
            font-family:'Segoe UI', sans-serif; font-size:15px; margin-top:10px;">
  <strong>What Happens in This Step:</strong>
  <ul style="margin-top:8px; margin-bottom:0;">
    <li>A new TEEHR evaluation folder is created (if it does not already exist).</li>
    <li>An empty <code>Evaluation</code> object is initialized using the TEEHR template structure.</li>
    <li>This directory will later store model outputs, observed data, alignment tables, 
        and evaluation metrics generated during the workflow.</li>
  </ul>
</div>


In [ ]:
%%time
teehr_dir = Path(SAMPLE_NGIAB_OUTPUT_DIR, "teehr")

if online == 'no':
    # Reuse the evaluation and the data already downloaded
    ev = teehr.Evaluation(dir_path=teehr_dir)
else:
    # First run: make a new evaluation from the empty template
    if teehr_dir.exists():
        shutil.rmtree(teehr_dir)
    ev = teehr.Evaluation(dir_path=teehr_dir, create_dir=True)
    ev.clone_template()

In [ ]:
# create df summarizing s3 evals
s3_df = ev.list_s3_evaluations()
s3_df.head()

In [ ]:
%%time
# get bucket url for e3
e3_row = s3_df[s3_df['name'] == 'e3_usgs_hourly_streamflow']
e3_url = e3_row['url'].values[0]

# format the url
e3_url = e3_url.replace("s3://", "s3a://")
s3_dataset_url = f"{e3_url}/dataset"
s3_crosswalk_url = f"{s3_dataset_url}/location_crosswalks/"

# query the location_crosswalk table from the s3 bucket ahead of cloning
s3_crosswalk_sdf = ev.spark.read.parquet(s3_crosswalk_url)

# convert to pandas for ease of use with your 'ngiab_gages_df'
s3_crosswalk_df = s3_crosswalk_sdf.toPandas()
s3_crosswalk_df.head(20)

<h4 style="background-color:rgba(184,34,61,0.18); padding:10px 14px; border-left: 5px solid #E04A63; border-radius:6px; font-family:'Segoe UI', sans-serif; font-size:17px; margin-top:18px;">
  🔹 Verify USGS Gage IDs in the Crosswalk Table
</h4>

<p style="font-family:'Segoe UI', sans-serif; line-height:1.55; margin-top:10px; ">
In this step, we verify that the USGS gage ID exist in the crosswalk table.
</p>

In [ ]:
usgs_gage = "usgs-02450250"

row = s3_crosswalk_df[s3_crosswalk_df["primary_location_id"] == usgs_gage]

print(row if not row.empty else "USGS gage not found")

<div style="background:#9E1B32; border-left:6px solid #F0627A; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#fff5f7;">
    6. Register USGS Gage Metadata and Define Location Crosswalk
  </h3>
</div>

<p style="margin-top:0; padding-top:6px;">
Our TEEHR evaluation starts empty, so we add the USGS gage (location and name from USGS NWIS) and the matching NextGen segment to it, and link them in a location crosswalk. This lets TEEHR download the USGS data and compare it with the NextGen simulation.
</p>


In [ ]:
%%time
# Register the USGS gage and its NGEN segment in TEEHR (only if it is not in the TEEHR list)

register_usgs_gage(ev, usgs_gage, ngiab_gages_df)

# Verify that the USGS gage has been registered in the location crosswalk table
ev.location_crosswalks.to_pandas()

<div style="background:#9E1B32; border-left:6px solid #F0627A; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#fff5f7;">
    7. Fetching USGS Streamgage Data
  </h3>
</div>
<p style="margin-top:0; padding-top:6px;">
We use TEEHR’s built-in tools to fetch USGS streamflow data
</p>

In [ ]:
%%time
# Fetch USGS streamflow observations for all registered USGS gages in the TEEHR
# (only when starting a new evaluation; 'offline' reuses the data already downloaded)

if online == 'yes':
    ev.fetch.usgs_streamflow(
        start_date=start_date,
        end_date=end_date
    )

In [ ]:
# Check registered TEEHR configurations
ev.configurations.to_pandas()

In [ ]:
# Convert primary timeseries table to a Pandas DataFrame
usgs_df = ev.primary_timeseries.to_pandas()

# Plot the time series using TEEHR's Pandas accessor
usgs_df.teehr.timeseries_plot()

<div style="background:#9E1B32; border-left:6px solid #F0627A; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#fff5f7;">
    8. Fetch NWM Streamflow Data
  </h3>
</div>
<p style="margin-top:0; padding-top:6px;">
We use TEEHR’s built-in fetching utilities to retrieve National Water Model (NWM) streamflow data for all <b>secondary location IDs</b> listed in the location crosswalk table 
that are prefixed with the selected NWM version (e.g., <code>nwm20</code>, <code>nwm21</code>, <code>nwm30</code>). 
This mechanism enables seamless alignment between observed (USGS) and modeled (NWM) streamflow at corresponding locations.
For additional details, see the TEEHR documentation:
<a href="https://rtiinternational.github.io/teehr/api/generated/teehr.evaluation.fetch.Fetch.nwm_retrospective_points.html" target="_blank">
NWM Retrospective Points Fetch
</a>.
</p>


<h4 style="background-color:rgba(184,34,61,0.18); padding:8px 12px; border-left: 5px solid #E04A63; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:16px; margin-top:16px;">
🔹 Register NWM feature_id in the Location Crosswalk Table
</h4>

<p style="font-family:'Segoe UI',sans-serif; font-size:15px; margin-top:6px;">
We link the <strong>USGS gage location</strong> to its corresponding  
<strong>National Water Model (NWM) feature</strong> by adding an entry to the  
<strong>TEEHR location crosswalk table</strong>.  
This crosswalk explicitly maps the USGS gage ID to the NWM retrospective  
<strong>feature_id (NHDPlus COMID)</strong>, enabling direct comparison between  
observed USGS streamflow and simulated NWM outputs within the TEEHR framework.
</p>


In [ ]:
%%time
nwm_feature_id = row.iloc[0, 1][6:] 

nwm_cw = pd.DataFrame([{
    "primary_location_id": usgs_gage,
    "secondary_location_id": f"nwm30-{nwm_feature_id}",
}])

ev.location_crosswalks.load_dataframe(nwm_cw)

<h4 style="background-color:rgba(184,34,61,0.18); padding:8px 12px; border-left: 5px solid #E04A63; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:16px; margin-top:16px;">
🔹 Fetch NWM Retrospective Streamflow Data
</h4>

<p style="font-family:'Segoe UI',sans-serif; font-size:15px; margin-top:6px;">
We use <strong>TEEHR’s built-in fetch utility</strong> to retrieve  
<strong>National Water Model (NWM) retrospective streamflow</strong> data for the  
specified simulation period.  

The fetch operation automatically uses the <strong> location crosswalk table to identify all secondary_location_id values prefixed with nwm30 and downloads the corresponding NWM streamflow time series. Then we register the fetched data as a primary dataset within the TEEHR evaluation environment, making it directly comparable to observed USGS streamflow.
</p>

In [ ]:
%%time
# If you want to download the data, you should use yes as input. For the workshop, we have downloaded it before. 

if online == "no":
    ev = teehr.Evaluation(dir_path=teehr_dir)
else:
    ev.fetch.nwm_retrospective_points(
        nwm_version="nwm30",
        variable_name="streamflow",
        start_date=start_date,
        end_date=end_date
    )

<h4 style="background-color:rgba(184,34,61,0.18); padding:8px 12px; border-left: 5px solid #E04A63; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:16px; margin-top:16px;">
🔹 Check the Crosswalk Table after Fetching the NWM Retrospective Streamflow Data
</h4>

In [ ]:
location_crosswalks_df = ev.location_crosswalks.to_pandas()
location_crosswalks_df

In [ ]:
usgs_df["value_time"] = pd.to_datetime(usgs_df["value_time"])
usgs_df = usgs_df.sort_values("value_time").reset_index(drop=True)
usgs_df

<div style="background:#9E1B32; border-left:6px solid #F0627A; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#fff5f7;">
    9. Retrieving Simulated Streamflow from NGIAB
  </h3>
</div>

<p style="margin-top:0; padding-top:6px;">
We retrieve simulated streamflow directly from the NGIAB <code>t-route</code> NetCDF output file and prepare it for subsequent evaluation and analysis workflows.
</p>

In [ ]:
#  Locate the NGIAB troute NetCDF output file
troute_output_file_list = list(
    Path(SAMPLE_NGIAB_OUTPUT_DIR, "outputs/troute").glob("*.nc")
)

# Select the first troute output file
troute_output_nc_filepath = troute_output_file_list[0]

# Open the troute routing dataset
troute_ds = xr.open_dataset(troute_output_nc_filepath)
print(troute_ds)

# Path where the temporary subset file will be saved
troute_subset_filepath = Path(TEMP_DIR, "troute_output_subset.nc")
print(troute_subset_filepath)

# Retrieve the validated NGIAB–USGS crosswalk from the Evaluation object
crosswalk_df = ev.location_crosswalks.to_pandas()
print(crosswalk_df)

# Extract the numeric NGIAB routing IDs (strip off the 'ngen-' prefix)
ngen_gages = [
    int(gage.split("-")[1])
    for gage in crosswalk_df.secondary_location_id.tolist()
    if gage.split("-")[0] == "ngen"
]

# Subset the troute dataset to the NGIAB routing segments of interest
troute_ds.sel(feature_id=ngen_gages).to_netcdf(troute_subset_filepath)

<h4 style="background-color:rgba(184,34,61,0.18); padding:10px 14px; border-left: 5px solid #E04A63; border-radius:6px; font-family:'Segoe UI', sans-serif; font-size:17px; margin-top:18px;">
  🔹 Add NextGen Configuration to the TEEHR Evaluation Environment
</h4>

In [ ]:
%%time
NGIAB_CONFIGURATION_NAME = "ngen_simulation"

# Get current configurations as a DataFrame
configs_df = ev.configurations.to_pandas()

# Check if the NGIAB configuration already exists
if NGIAB_CONFIGURATION_NAME not in configs_df["name"].values:
    ev.configurations.add([
        teehr.Configuration(
            name=NGIAB_CONFIGURATION_NAME,
            type="secondary",
            description="Nextgen simulation output"
        )
    ])
else:
    print(f"Configuration '{NGIAB_CONFIGURATION_NAME}' already exists — skipping add.")

<h4 style="background-color:rgba(184,34,61,0.18); padding:10px 14px; border-left: 5px solid #E04A63; border-radius:6px; font-family:'Segoe UI', sans-serif; font-size:17px; margin-top:18px;">
  🔹 Review the Current Configurations
</h4>
<p style="font-family:'Segoe UI', sans-serif; line-height:1.55; margin-top:10px; ">
Before running the evaluation, it is useful to review the configuration settings currently stored inside the <code>Evaluation</code> object.
The next cell uses <code>ev.configurations.to_pandas()</code> to display these settings as a Pandas DataFrame.

In [ ]:
%%time
# Display the configuration settings as a Pandas DataFrame
ev.configurations.to_pandas()

<p style="font-family:'Segoe UI', sans-serif; line-height:1.6; margin-top:10px;">
The configuration table now shows three registered datasets inside the <code>Evaluation</code> object.
Each entry defines a data source used during the evaluation process:
</p>
<ul style="font-family:'Segoe UI', sans-serif; line-height:1.6; ">
  <li><strong>nwm30_retrospective</strong> — a <em>secondary</em> dataset containing the NWM 3.0 retrospective model outputs.</li>
  <li><strong>usgs_observations</strong> — the <em>primary</em> dataset, representing the USGS observed streamflow.</li>
  <li><strong>gage_024654931</strong> — an additional <em>secondary</em> dataset describing the NGIAB (NextGen) simulation output for the selected gage.</li>
</ul>

<h4 style="background-color:rgba(184,34,61,0.18); padding:10px 14px; border-left: 5px solid #E04A63;
           border-radius:6px; font-family:'Segoe UI', sans-serif; font-size:17px; margin-top:22px;">
  🔹 Load the NGIAB Streamflow Time Series into the Evaluation
</h4>
<p style="font-family:'Segoe UI', sans-serif; line-height:1.6; margin-top:10px; ">
We load the NGIAB simulated streamflow data from the subsetted
<code>troute</code> NetCDF file into the <code>Evaluation</code> object.  
This registers the NGIAB time series as a <strong>secondary dataset</strong>, allowing TEEHR to align it later with
the USGS observations for comparison.
</p>
<p style="font-family:'Segoe UI', sans-serif; line-height:1.6; margin-top:10px;">
The <code>field_mapping</code> tells TEEHR how to interpret the NetCDF fields, while
<code>constant_field_values</code> provides metadata such as units, variable names,
and the configuration associated with this dataset.
Using <code>location_id_prefix="ngen"</code> ensures that location IDs in the time series
match the crosswalk entries created earlier.
</p>

In [ ]:
%%time
ev.secondary_timeseries.load_netcdf(
    in_path=troute_subset_filepath,
    field_mapping={
        "time": "value_time",
        "feature_id": "location_id",
        "flow": "value"
    },
    constant_field_values={
        "unit_name": "m^3/s",
        "variable_name": "streamflow_hourly_inst",
        "configuration_name": NGIAB_CONFIGURATION_NAME,
        "reference_time": None
    },
    location_id_prefix="ngen"
)

<div style="background:#9E1B32; border-left:6px solid #F0627A; padding:14px 18px; border-radius:8px; margin-top:0; margin-bottom:12px;">
  <h3 style="margin:0; font-size:21px; font-weight:700; color:#fff5f7; font-family:'Segoe UI', sans-serif;">
    10. Visualize NGIAB, NWM, and USGS Streamflow Time Series
  </h3>
</div>
<p style="font-family:'Segoe UI', sans-serif; line-height:1.65; margin-top:8px;">
In this step, we extract and plot the streamflow time series from all three datasets used in the evaluation:
</p>
<ul style="font-family:'Segoe UI', sans-serif; line-height:1.6; margin-top:6px;">
  <li><strong>NGIAB (NextGen)</strong> simulation output for the selected routing segment</li>
  <li><strong>NWM 3.0 retrospective</strong> model output for the corresponding segment</li>
  <li><strong>USGS</strong> observed streamflow for the matching gage</li>
</ul>

In [ ]:
# Streamflow: USGS observed vs NWM vs NextGen
plot_teehr_streamflow(ev, usgs_gage, nwm_feature_id, ngiab_gages_df.iloc[0, 0])

In [ ]:
%%time
# Join the observed (primary) and simulated (secondary) series on location and time,
# so every model value sits next to the matching USGS value
ev.joined_timeseries.create(add_attrs=False)


In [ ]:
# Compute the metrics for each location and each model (configuration) against USGS observations
metrics_df = ev.metrics.query(
    group_by=["primary_location_id", "configuration_name"],
    include_metrics=[
        dm.KlingGuptaEfficiency(),
        dm.NashSutcliffeEfficiency(),
        dm.RootMeanSquareError(),
        dm.RelativeBias(),
    ],
).to_pandas()

# Relative bias as a percent (PBIAS)
metrics_df["pbias_percent"] = metrics_df["relative_bias"] * 100
metrics_df


[**LETS GO TO THE NEXT PART**](./05.extreme_events_.ipynb)